# 00 Part 2 — Image Download — Women Shoes Size 8 (Clean ASINs Only)

**Input :** Train and val parquets from data preparation (5,147 clean ASINs)

**Output:** `data/images/{ASIN}.jpg` — one image per unique ASIN

**Notes:**
- Only downloads images for ASINs that passed the full data preparation pipeline
- Safe to re-run — already downloaded images are skipped automatically
- ~5,147 images, estimated 500–700 MB total
- Takes 20–45 minutes on CPU — no GPU needed
- Run AFTER `00_part1_data_preparation.ipynb` is complete

## ① Setup

In [ ]:
# Dependencies: pyarrow, pandas, requests, Pillow, tqdm (install locally)
print('Local mode')

## ② Config

In [ ]:
import os
from pathlib import Path

# Local mode - no Google Drive needed
_cwd = Path.cwd()
PROJECT_ROOT = _cwd
for _ in range(5):
    if (PROJECT_ROOT / 'data').is_dir() and (PROJECT_ROOT / 'code').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Cannot find project root (expected 'data/' and 'code/' folders)")

ROOT = str(PROJECT_ROOT) + '/'
print(f'PROJECT_ROOT: {PROJECT_ROOT}')

DATA_DIR = ROOT + 'data/amzn_shoes_monthly_diffs_ffill_fixed_splits/'
IMG_DIR  = ROOT + 'data/images/'

TRAIN_FILE = DATA_DIR + 'train-00000-of-00001.parquet'
VAL_FILE   = DATA_DIR + 'validation-00000-of-00001.parquet'

MAX_RETRIES = 3
TIMEOUT_SEC = 10
SLEEP_SEC   = 0.2

os.makedirs(IMG_DIR, exist_ok=True)

print(f'Image output dir: {IMG_DIR}')

## ③ Load Clean ASINs and Image URLs from Saved Parquets

In [ ]:
import pandas as pd

# Load both splits — these are the ONLY clean ASINs we want images for
df_train = pd.read_parquet(TRAIN_FILE, columns=['ASIN', 'image'])
df_val   = pd.read_parquet(VAL_FILE,   columns=['ASIN', 'image'])

df_all = pd.concat([df_train, df_val], ignore_index=True)

# One row per ASIN — image URL is static per product
asin_urls = (
    df_all[['ASIN', 'image']]
    .drop_duplicates('ASIN')
    .dropna(subset=['image'])
    .reset_index(drop=True)
)

print(f'Train ASINs        : {df_train["ASIN"].nunique():,}')
print(f'Val ASINs          : {df_val["ASIN"].nunique():,}')
print(f'Total unique ASINs : {asin_urls["ASIN"].nunique():,}')
print(f'ASINs with image URL : {len(asin_urls):,}')
print(f'ASINs missing image  : {df_all["ASIN"].nunique() - len(asin_urls):,}')
print()
print('Sample URLs:')
for _, row in asin_urls.head(3).iterrows():
    print(f'  {row["ASIN"]} → {row["image"]}')

## ④ Check Already Downloaded Images

In [ ]:
already_done = set()
if os.path.exists(IMG_DIR):
    for fname in os.listdir(IMG_DIR):
        if fname.endswith('.jpg'):
            already_done.add(fname.replace('.jpg', ''))

to_download = asin_urls[~asin_urls['ASIN'].isin(already_done)].reset_index(drop=True)

print(f'Already downloaded : {len(already_done):,} images')
print(f'To download        : {len(to_download):,} images')
print(f'Target total       : {len(asin_urls):,} images')

## ⑤ Download Images

In [ ]:
import requests
import time
from PIL import Image
from io import BytesIO
from tqdm.notebook import tqdm

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )
}

success  = []
failed   = []
skipped  = list(already_done)

for _, row in tqdm(to_download.iterrows(), total=len(to_download), desc='Downloading'):
    asin      = row['ASIN']
    url       = row['image']
    save_path = IMG_DIR + f'{asin}.jpg'

    downloaded = False
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.get(url, headers=HEADERS, timeout=TIMEOUT_SEC)
            if response.status_code == 200:
                img = Image.open(BytesIO(response.content))
                img = img.convert('RGB')
                img.save(save_path, 'JPEG')
                success.append(asin)
                downloaded = True
                break
            else:
                time.sleep(1)
        except Exception:
            time.sleep(1)

    if not downloaded:
        failed.append({'ASIN': asin, 'url': url})

    time.sleep(SLEEP_SEC)

print(f'\n✅ Downloaded : {len(success):,}')
print(f'⏭  Skipped    : {len(skipped):,}  (already existed)')
print(f'❌ Failed     : {len(failed):,}')

## ⑥ Failed Downloads — Review and Retry if Needed

In [ ]:
if len(failed) > 0:
    print(f'Failed ASINs ({len(failed)} total):')
    for item in failed:
        print(f'  {item["ASIN"]} → {item["url"]}')
    print()
    pct_failed = len(failed) / len(asin_urls) * 100
    if pct_failed < 5:
        print(f'Failure rate: {pct_failed:.1f}% — acceptable. Re-run this notebook to retry.')
    else:
        print(f'Failure rate: {pct_failed:.1f}% — higher than expected. Check internet connection.')
else:
    print('✅ All images downloaded successfully — no failures!')

## ⑦ Final Summary

In [ ]:
from PIL import Image as PILImage

final_count = len([f for f in os.listdir(IMG_DIR) if f.endswith('.jpg')])
total_size  = sum(
    os.path.getsize(IMG_DIR + f)
    for f in os.listdir(IMG_DIR)
    if f.endswith('.jpg')
) / 1e6

print('=' * 55)
print('IMAGE DOWNLOAD SUMMARY')
print('=' * 55)
print(f'Images on disk  : {final_count:,}')
print(f'Target          : {len(asin_urls):,}')
print(f'Coverage        : {final_count/len(asin_urls)*100:.1f}%')
print(f'Total size      : {total_size:.1f} MB')
print(f'Location        : {IMG_DIR}')
print()

# Verify a few image dimensions
sample_files = [f for f in os.listdir(IMG_DIR) if f.endswith('.jpg')][:5]
print('Sample image sizes:')
for fname in sample_files:
    img = PILImage.open(IMG_DIR + fname)
    print(f'  {fname.replace(".jpg","")} : {img.size[0]} x {img.size[1]} px')

print()

# Cross-check: confirm all clean ASINs have an image
clean_asins   = set(asin_urls['ASIN'].unique())
images_on_disk = set(f.replace('.jpg','') for f in os.listdir(IMG_DIR) if f.endswith('.jpg'))
missing_images = clean_asins - images_on_disk

if len(missing_images) == 0:
    print(f'✅ All {len(clean_asins):,} clean ASINs have an image on disk.')
else:
    print(f'⚠️  {len(missing_images)} clean ASINs still missing images:')
    for asin in sorted(missing_images)[:20]:
        print(f'  {asin}')
    if len(missing_images) > 20:
        print(f'  ... and {len(missing_images)-20} more')
    print('Re-run from Cell ⑤ to retry.')

print()
if final_count >= len(asin_urls) * 0.95:
    print('✅ Ready for embedding training!')
    print()
    print('Next step: run 01_1_create_dataset_txt_img.ipynb (GPU required)')
else:
    print('⚠️  Coverage below 95% — re-run Cell ⑤ to retry failed downloads.')